In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "smoke"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage3"
SEED = 71
MODEL_NAME = [
    "dino_wm_pusht",
    "jepa_wm_pusht",
    "dino_wm_wall",
    "jepa_wm_wall",
]
ENVIRONMENT = ["PushT", "Wall"]
HORIZONS = [1, 3, 6]
NUM_STATES = 36  # per environment in smoke mode
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage3"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
FEATURE_POOL_GRID = 4
TASKS_PER_ENVIRONMENT = 12
TASK_SPLIT_COUNTS = [6, 2, 2, 2]  # probe train, calibration, regression train, final test
EVALUATION_SEEDS = [71, 131, 191]
PROBE_SEEDS = [2071]
READOUT_PROJECTION_DIM = 256
RIDGE_LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
REGRESSION_RIDGE = 1e-3
BOOTSTRAP_REPS = 300
RANKING_TIE = 1e-9

if RUN_MODE == "full":
    NUM_STATES = 240  # 20 states per task, per environment
    PROBE_SEEDS = [2071, 4071, 6071]
    BOOTSTRAP_REPS = 2000
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == [
    "dino_wm_pusht",
    "jepa_wm_pusht",
    "dino_wm_wall",
    "jepa_wm_wall",
]
assert ENVIRONMENT == ["PushT", "Wall"]
assert HORIZONS == [1, 3, 6]
assert 8 <= ACTIONS_PER_STATE <= 12
assert NUM_STATES % TASKS_PER_ENVIRONMENT == 0
assert sum(TASK_SPLIT_COUNTS) == TASKS_PER_ENVIRONMENT


# Stage 3: full counterfactual benchmark

This notebook tests whether the Stage 2C task-aligned signal generalizes beyond
PushT. It runs paired executable interventions in **PushT and Wall**, evaluates
four public environment-specific checkpoints (DINO-WM and JEPA-WM in each
environment), and holds out both simulator states and goal/layout tasks.

The primary comparison is a frozen **linear physical-state readout** against raw
latent-goal distance. The readout is fit on probe-train tasks/states, selected
only by calibration pose error, and evaluated on untouched final tasks/states.

A second, pre-specified test fits a regression on separate regression-train
tasks/states and asks whether counterfactual margin error improves final-test
planning-regret prediction beyond ordinary endpoint/cost error.

Exact restoration, fixed candidate libraries, action-blind/shuffled/oracle
controls, contact/collision strata, clustered bootstrap intervals, resumable
shards, and automatic ZIP download are included. Only simulator claims are
licensed; this does not establish real-robot reliability.

The public checkpoint release provides one training seed per environment/model
configuration. We therefore report checkpoint-seed breadth as unavailable and
use three evaluation seeds plus three readout-projection seeds in the full run.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
INTERMEDIATE = OUT / "intermediate"
TRUTH_ROOT = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
PROBE_DIR = OUT / "probes"
for path in [
    OUT,
    INTERMEDIATE,
    TRUTH_ROOT,
    MODEL_ROOT,
    LOG_DIR,
    PLOT_DIR,
    PROBE_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage3")
log.info("Seeds set to %d; evaluation seeds=%s; probe seeds=%s", SEED, EVALUATION_SEEDS, PROBE_SEEDS)


def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload


CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "FEATURE_POOL_GRID": FEATURE_POOL_GRID,
    "TASKS_PER_ENVIRONMENT": TASKS_PER_ENVIRONMENT,
    "TASK_SPLIT_COUNTS": TASK_SPLIT_COUNTS,
    "EVALUATION_SEEDS": EVALUATION_SEEDS,
    "PROBE_SEEDS": PROBE_SEEDS,
    "READOUT_PROJECTION_DIM": READOUT_PROJECTION_DIM,
    "RIDGE_LAMBDAS": RIDGE_LAMBDAS,
    "REGRESSION_RIDGE": REGRESSION_RIDGE,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "RANKING_TIE": RANKING_TIE,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


gpu_report("startup")


In [ ]:
MODEL_BY_ENVIRONMENT = {
    "PushT": ["dino_wm_pusht", "jepa_wm_pusht"],
    "Wall": ["dino_wm_wall", "jepa_wm_wall"],
}
READOUTS = [
    "latent_distance",
    "linear_pose",
    "action_blind",
    "linear_pose_shuffled",
    "oracle_pose",
]
SPLIT_NAMES = [
    "probe_train",
    "probe_calibration",
    "regression_train",
    "final_test",
]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=True) + "\n")


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError(f"cannot infer fields for empty table {path}")
        fieldnames = list(rows[0])
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def pair_indices(n_actions):
    return np.triu_indices(n_actions, k=1)


def pool_visual(visual, grid=FEATURE_POOL_GRID):
    value = visual.detach().float().cpu().numpy()[..., 0, :, :, :]
    height, width, dim = value.shape[-3:]
    if height % grid or width % grid:
        raise ValueError(f"feature grid {(height, width)} is not divisible by {grid}")
    fh, fw = height // grid, width // grid
    value = value.reshape(*value.shape[:-3], grid, fh, grid, fw, dim)
    value = value.mean(axis=(-4, -2))
    return value.reshape(*value.shape[:-3], -1)


def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm


def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    return np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    ) @ np.asarray(vector)


def feature_metrics(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    truth_delta = truth[left] - truth[right]
    predicted_delta = prediction[left] - prediction[right]
    pair_error = predicted_delta - truth_delta
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    pair_scale = np.sqrt(np.mean(truth_delta**2, axis=(0, 2)))
    normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(truth_delta * predicted_delta, axis=-1)
    denominator = (
        np.linalg.norm(truth_delta, axis=-1)
        * np.linalg.norm(predicted_delta, axis=-1)
    )
    cosine = np.divide(
        dot,
        denominator,
        out=np.zeros_like(dot),
        where=denominator > eps,
    ).mean(axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    return {
        "ordinary_feature_rmse": ordinary,
        "common_mode_feature_rmse": np.sqrt(np.mean(common**2, axis=-1)),
        "action_dependent_feature_rmse": action_dependent,
        "paired_feature_rmse": pair_rmse,
        "normalized_paired_feature_rmse": normalized,
        "paired_feature_cosine": cosine,
        "pair_identity_residual": pair_rmse**2 - expected_pair_mse,
    }


def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = (
        float(np.sqrt(np.mean(true_margin[valid] ** 2))) if np.any(valid) else 0.0
    )
    normalized_margin_rmse = (
        float(
            np.sqrt(
                np.mean((predicted_margin[valid] - true_margin[valid]) ** 2)
            )
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
        "pair_left": left,
        "pair_right": right,
        "true_margin": true_margin,
        "predicted_margin": predicted_margin,
        "pair_credit": credit,
        "pair_weight": weights,
    }


def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": repetitions,
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }


def random_projection(input_dim, output_dim, seed):
    rng = np.random.default_rng(seed)
    return (
        rng.standard_normal((input_dim, output_dim)).astype(np.float32)
        / np.sqrt(output_dim)
    )


def standardize_fit(values):
    values = np.asarray(values, dtype=np.float64)
    mean = np.mean(values, axis=0)
    scale = np.std(values, axis=0)
    scale[scale < 1e-8] = 1.0
    return mean, scale


def fit_linear_readout(x_train, y_train, x_calibration, y_calibration):
    mean, scale = standardize_fit(x_train)
    train = (np.asarray(x_train, dtype=np.float64) - mean) / scale
    calibration = (np.asarray(x_calibration, dtype=np.float64) - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    calibration = np.column_stack([np.ones(len(calibration)), calibration])
    gram = train.T @ train
    cross = train.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in RIDGE_LAMBDAS:
        penalty = np.eye(gram.shape[0]) * ridge
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_pose_mse": loss,
            "mean": mean,
            "scale": scale,
            "coefficient": coefficient,
        }
        if best is None or loss < best["calibration_pose_mse"]:
            best = candidate
    return best


def predict_linear_readout(probe, values):
    standardized = (
        np.asarray(values, dtype=np.float64) - probe["mean"]
    ) / probe["scale"]
    augmented = np.column_stack([np.ones(len(standardized)), standardized])
    return augmented @ probe["coefficient"]


def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT],
        check=True,
    )
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def task_split_map():
    rng = np.random.default_rng(SEED + 17)
    order = rng.permutation(TASKS_PER_ENVIRONMENT).tolist()
    mapping = {}
    start = 0
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS):
        for task_id in order[start : start + count]:
            mapping[int(task_id)] = name
        start += count
    return mapping


def pusht_tasks():
    values = [
        (210.0, 210.0, -0.75),
        (210.0, 256.0, 0.00),
        (210.0, 302.0, 0.75),
        (256.0, 210.0, 0.60),
        (256.0, 256.0, np.pi / 4),
        (256.0, 302.0, -0.60),
        (302.0, 210.0, 0.00),
        (302.0, 256.0, np.pi / 4),
        (302.0, 302.0, -np.pi / 4),
        (232.0, 232.0, 1.10),
        (280.0, 232.0, -1.10),
        (256.0, 280.0, 0.30),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "PushT",
            "task_id": index,
            "task_name": f"pusht_goal_{index:02d}",
            "goal": list(value),
            "split": splits[index],
        }
        for index, value in enumerate(values)
    ]


def wall_tasks():
    layouts = [
        (27.0, 18.0, 53.0, 46.0),
        (27.0, 30.0, 53.0, 18.0),
        (27.0, 42.0, 53.0, 34.0),
        (32.0, 18.0, 12.0, 46.0),
        (32.0, 30.0, 12.0, 18.0),
        (32.0, 42.0, 12.0, 34.0),
        (37.0, 18.0, 53.0, 30.0),
        (37.0, 30.0, 53.0, 46.0),
        (37.0, 42.0, 53.0, 18.0),
        (29.0, 22.0, 12.0, 42.0),
        (34.0, 34.0, 53.0, 22.0),
        (36.0, 40.0, 12.0, 26.0),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "Wall",
            "task_id": index,
            "task_name": f"wall_layout_goal_{index:02d}",
            "wall_x": wall_x,
            "door_y": door_y,
            "goal": [goal_x, goal_y],
            "split": splits[index],
        }
        for index, (wall_x, door_y, goal_x, goal_y) in enumerate(layouts)
    ]


TASKS = {"PushT": pusht_tasks(), "Wall": wall_tasks()}


def build_state_records(environment):
    tasks = TASKS[environment]
    per_task = NUM_STATES // TASKS_PER_ENVIRONMENT
    records = []
    state_id = 0
    for task in tasks:
        for within_task in range(per_task):
            evaluation_seed = EVALUATION_SEEDS[within_task % len(EVALUATION_SEEDS)]
            rng = np.random.default_rng(
                evaluation_seed * 100000
                + task["task_id"] * 1000
                + within_task
            )
            if environment == "PushT":
                goal_xy = np.asarray(task["goal"][:2], dtype=np.float64)
                for _ in range(100):
                    radial = rng.uniform(85.0, 120.0)
                    polar = rng.uniform(-np.pi, np.pi)
                    block = goal_xy + radial * np.array(
                        [np.cos(polar), np.sin(polar)]
                    )
                    direction = unit_vector(goal_xy - block)
                    agent_distance = rng.uniform(58.0, 80.0)
                    agent = block - agent_distance * direction
                    if (
                        np.all(block > 90.0)
                        and np.all(block < 422.0)
                        and np.all(agent > 35.0)
                        and np.all(agent < 477.0)
                    ):
                        break
                else:
                    raise RuntimeError("could not build bounded PushT state")
                state = np.array(
                    [
                        agent[0],
                        agent[1],
                        block[0],
                        block[1],
                        rng.uniform(-0.65, 0.65),
                        0.0,
                        0.0,
                    ],
                    dtype=np.float64,
                )
                stratum = "near" if agent_distance < 69.0 else "far"
            else:
                wall_x = float(task["wall_x"])
                goal_x = float(task["goal"][0])
                goal_on_right = goal_x > wall_x
                if goal_on_right:
                    x = rng.uniform(8.0, max(9.0, wall_x - 8.0))
                else:
                    x = rng.uniform(min(56.0, wall_x + 8.0), 57.0)
                y = rng.uniform(8.0, 57.0)
                state = np.array([x, y], dtype=np.float64)
                stratum = "left_to_right" if goal_on_right else "right_to_left"
            records.append(
                {
                    "environment": environment,
                    "state_id": state_id,
                    "task_id": task["task_id"],
                    "task_name": task["task_name"],
                    "split": task["split"],
                    "evaluation_seed": int(evaluation_seed),
                    "design_stratum": stratum,
                    "state": state,
                }
            )
            state_id += 1
    if len(records) != NUM_STATES:
        raise AssertionError("state-record count mismatch")
    return records


def pusht_candidate_library(state, task, primitive_steps):
    direction = unit_vector(
        np.asarray(task["goal"][:2]) - np.asarray(state)[2:4]
    )
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    selected = np.asarray(
        [0, 2, 4, 6, 8, 11, 14, 16, 17, 18],
        dtype=np.int64,
    )
    return (
        np.stack(sequences)[selected],
        [specifications[index][0] for index in selected],
        selected,
    )


def nominal_waypoint_sequence(state, waypoints, primitive_steps, magnitude=0.75):
    position = np.asarray(state, dtype=np.float64).copy()
    sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
    waypoints = [np.asarray(point, dtype=np.float64) for point in waypoints]
    for step in range(primitive_steps):
        remaining = primitive_steps - step
        waypoint_index = min(
            len(waypoints) - 1,
            (step * len(waypoints)) // primitive_steps,
        )
        delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) < 0.5 and waypoint_index + 1 < len(waypoints):
            waypoint_index += 1
            delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) > 1e-8:
            action = magnitude * unit_vector(delta)
            sequence[step] = action.astype(np.float32)
            position = position + 2.0 * action
    return sequence


def wall_candidate_library(state, task, primitive_steps):
    state = np.asarray(state, dtype=np.float64)
    goal = np.asarray(task["goal"], dtype=np.float64)
    wall_x = float(task["wall_x"])
    door_y = float(task["door_y"])
    side = np.sign(goal[0] - state[0])
    door = np.array([wall_x + side * 1.0, door_y])
    directions = [
        ("noop", np.zeros((primitive_steps, 2), dtype=np.float32)),
        ("direct", nominal_waypoint_sequence(state, [goal], primitive_steps)),
        (
            "via_door",
            nominal_waypoint_sequence(state, [door, goal], primitive_steps),
        ),
        (
            "via_door_high",
            nominal_waypoint_sequence(
                state,
                [door + np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
        (
            "via_door_low",
            nominal_waypoint_sequence(
                state,
                [door - np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
    ]
    direct = unit_vector(goal - state)
    for angle in [-35.0, 35.0, -70.0, 70.0]:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        sequence[:] = (0.75 * rotate_vector(direct, angle)).astype(np.float32)
        directions.append((f"angle_{angle:+.0f}", sequence))
    reverse = np.zeros((primitive_steps, 2), dtype=np.float32)
    reverse[:] = (-0.55 * direct).astype(np.float32)
    directions.append(("reverse", reverse))
    if len(directions) != ACTIONS_PER_STATE:
        raise AssertionError("Wall candidate count mismatch")
    return (
        np.stack([item[1] for item in directions]),
        [item[0] for item in directions],
        np.arange(ACTIONS_PER_STATE, dtype=np.int64),
    )


def candidate_library(environment, state, task, primitive_steps):
    if environment == "PushT":
        return pusht_candidate_library(state, task, primitive_steps)
    return wall_candidate_library(state, task, primitive_steps)


def task_cost(environment, states, task):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle_error = np.arctan2(
            np.sin(states[..., 4] - goal[2]),
            np.cos(states[..., 4] - goal[2]),
        )
        pieces = np.concatenate(
            [
                (states[..., 2:4] - goal[:2]) / 512.0,
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    goal = np.asarray(task["goal"], dtype=np.float64)
    return np.linalg.norm((states[..., :2] - goal) / 65.0, axis=-1)


def pose_target(environment, states):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        angle = states[..., 4]
        return np.stack(
            [
                states[..., 2] / 512.0,
                states[..., 3] / 512.0,
                np.sin(angle),
                np.cos(angle),
            ],
            axis=-1,
        )
    return states[..., :2] / 65.0


def decoded_task_cost(environment, prediction, task):
    prediction = np.asarray(prediction, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_error = np.arctan2(
            np.sin(angle - goal[2]),
            np.cos(angle - goal[2]),
        )
        return np.linalg.norm(
            np.concatenate(
                [
                    prediction[..., :2] - goal[:2] / 512.0,
                    (angle_error / np.pi)[..., None],
                ],
                axis=-1,
            ),
            axis=-1,
        )
    return np.linalg.norm(
        prediction[..., :2] - np.asarray(task["goal"]) / 65.0,
        axis=-1,
    )


def physical_pose_error(environment, prediction, truth):
    prediction = np.asarray(prediction, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if environment == "PushT":
        angle_prediction = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_truth = np.arctan2(truth[..., 2], truth[..., 3])
        angle_error = np.arctan2(
            np.sin(angle_prediction - angle_truth),
            np.cos(angle_prediction - angle_truth),
        )
        pieces = np.concatenate(
            [
                prediction[..., :2] - truth[..., :2],
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    return np.linalg.norm(prediction[..., :2] - truth[..., :2], axis=-1)


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        task = TASKS["Wall"][0]
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def wall_visual(env):
    value = env.render().float()[None]
    resized = torch_functional.interpolate(
        value,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )[0]
    return (
        torch.clamp(torch.round(resized), 0, 255)
        .to(torch.uint8)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


def reset_environment(repo, environment, task, state, seed):
    if environment == "PushT":
        env = make_environment(repo, environment)
        env.seed(seed)
        env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
        observation, restored = env.reset()
        payload = {
            "visual": np.asarray(observation["visual"]).copy(),
            "proprio": np.asarray(observation["proprio"]).copy(),
        }
        return env, payload, np.asarray(restored).copy()

    env = make_environment(repo, environment, task)
    env.seed(seed)
    env.reset_to_state = torch.as_tensor(
        np.asarray(state, dtype=np.float32)
    )
    observation, restored = env.reset()
    payload = {
        "visual": wall_visual(env),
        "proprio": np.asarray(
            observation["proprio"].detach().cpu(), dtype=np.float32
        ),
    }
    return env, payload, np.asarray(restored.detach().cpu(), dtype=np.float32)


def rollout_branch(repo, environment, task, state, actions, seed):
    env, initial, restored = reset_environment(
        repo, environment, task, state, seed
    )
    wanted = set(HORIZONS)
    observations = {}
    states = {}
    interactions = {}
    interaction_types = {}
    cumulative_interactions = 0
    cumulative_crossings = 0
    previous_state = np.asarray(restored, dtype=np.float64).copy()

    for step, action in enumerate(actions, start=1):
        if environment == "PushT":
            observation, _, _, info = env.step(action)
            current_state = np.asarray(info["state"]).copy()
            cumulative_interactions += int(info.get("n_contacts", 0))
            current_observation = {
                "visual": np.asarray(observation["visual"]).copy(),
                "proprio": np.asarray(observation["proprio"]).copy(),
            }
        else:
            observation, _, _, info = env.step(
                torch.as_tensor(action, dtype=torch.float32)
            )
            current_state = np.asarray(
                info["state"].detach().cpu(), dtype=np.float64
            )
            proposed = previous_state + 2.0 * np.asarray(action)
            if np.linalg.norm(current_state - proposed) > 1e-5:
                cumulative_interactions += 1
            wall_x = float(task["wall_x"])
            crossed = (
                (previous_state[0] - wall_x) * (current_state[0] - wall_x)
                < 0
            )
            cumulative_crossings += int(crossed)
            current_observation = {
                "visual": wall_visual(env),
                "proprio": np.asarray(
                    observation["proprio"].detach().cpu(),
                    dtype=np.float32,
                ),
            }
        previous_state = current_state.copy()
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = current_observation
                states[horizon] = current_state
                interactions[horizon] = cumulative_interactions
                if environment == "PushT":
                    interaction_types[horizon] = (
                        "contact" if cumulative_interactions > 0 else "free"
                    )
                elif cumulative_interactions > 0:
                    interaction_types[horizon] = "collision"
                elif cumulative_crossings > 0:
                    interaction_types[horizon] = "door_cross"
                else:
                    interaction_types[horizon] = "free"
    if wanted != set(observations):
        raise RuntimeError(f"missing horizons: {wanted - set(observations)}")
    return (
        initial,
        restored,
        observations,
        states,
        interactions,
        interaction_types,
    )


def exact_restore_test(repo, environment, task, state, actions):
    endpoints = []
    images = []
    interactions = []
    for _ in range(3):
        initial, _, _, states, counts, kinds = rollout_branch(
            repo,
            environment,
            task,
            state,
            actions,
            SEED + 9000,
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        interactions.append(
            (counts[max(HORIZONS)], kinds[max(HORIZONS)])
        )
    result = {
        "environment": environment,
        "repeats": 3,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            interactions[0] == item for item in interactions[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(
                np.max(np.abs(endpoints[0] - item))
                for item in endpoints[1:]
            )
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result


def goal_observation(repo, environment, task):
    if environment == "PushT":
        goal = task["goal"]
        state = np.array(
            [80.0, 450.0, goal[0], goal[1], goal[2], 0.0, 0.0]
        )
    else:
        state = np.asarray(task["goal"], dtype=np.float64)
    _, observation, _ = reset_environment(
        repo,
        environment,
        task,
        state,
        SEED + 11000 + task["task_id"],
    )
    return observation


In [ ]:
# Phase A — exact paired interventions in PushT and Wall.
def generate_simulator_truth():
    repo = configure_repo()
    task_payload = []
    split_payload = {
        "protocol": (
            "task-disjoint probe-train/calibration/regression-train/final-test; "
            "states are nested within exactly one task"
        ),
        "environments": {},
    }
    restore_payload = {}
    design_payload = {}

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        truth_dir.mkdir(parents=True, exist_ok=True)
        records = build_state_records(environment)
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for task in TASKS[environment]:
            task_payload.append(task)

        split_payload["environments"][environment] = {
            name: {
                "task_ids": sorted(
                    task["task_id"]
                    for task in TASKS[environment]
                    if task["split"] == name
                ),
                "state_ids": sorted(
                    record["state_id"]
                    for record in records
                    if record["split"] == name
                ),
            }
            for name in SPLIT_NAMES
        }

        primitive_steps = max(HORIZONS) * FRAMESKIP
        first = records[0]
        first_task = tasks_by_id[first["task_id"]]
        first_actions, labels, selected = candidate_library(
            environment,
            first["state"],
            first_task,
            primitive_steps,
        )
        restore_payload[environment] = exact_restore_test(
            repo,
            environment,
            first_task,
            first["state"],
            first_actions[1],
        )

        state_matrix = []
        task_ids = []
        action_bank = []
        physical_costs = []
        interaction_bank = []
        interaction_type_bank = []
        for record in records:
            state_id = record["state_id"]
            state_path = truth_dir / f"state_{state_id:04d}.npz"
            if state_path.exists():
                log.info("%s simulator resume: keeping %s", environment, state_path.name)
                with np.load(state_path) as shard:
                    state_matrix.append(shard["initial_state"])
                    task_ids.append(int(shard["task_id"]))
                    action_bank.append(shard["selected_actions"])
                    physical_costs.append(shard["physical_cost"])
                    interaction_bank.append(shard["interactions"])
                    interaction_type_bank.append(shard["interaction_types"])
                continue

            task = tasks_by_id[record["task_id"]]
            actions, action_labels, selected_indices = candidate_library(
                environment,
                record["state"],
                task,
                primitive_steps,
            )
            if action_labels != labels:
                raise AssertionError("candidate labels changed across states")
            initials = []
            visuals = []
            proprios = []
            endpoints = []
            interactions = []
            interaction_types = []
            for branch in actions:
                (
                    initial,
                    _,
                    observations,
                    states,
                    counts,
                    kinds,
                ) = rollout_branch(
                    repo,
                    environment,
                    task,
                    record["state"],
                    branch,
                    record["evaluation_seed"] * 1000 + state_id,
                )
                initials.append(initial["visual"])
                visuals.append(
                    [observations[horizon]["visual"] for horizon in HORIZONS]
                )
                proprios.append(
                    [observations[horizon]["proprio"] for horizon in HORIZONS]
                )
                endpoints.append(
                    [states[horizon] for horizon in HORIZONS]
                )
                interactions.append(
                    [counts[horizon] for horizon in HORIZONS]
                )
                interaction_types.append(
                    [kinds[horizon] for horizon in HORIZONS]
                )
            if not all(
                np.array_equal(initials[0], item) for item in initials[1:]
            ):
                raise AssertionError(
                    f"branch initial render mismatch: {environment} state {state_id}"
                )
            endpoint_array = np.asarray(endpoints, dtype=np.float32)
            physical_cost = task_cost(
                environment, endpoint_array, task
            ).astype(np.float32)
            _, initial_observation, _ = reset_environment(
                repo,
                environment,
                task,
                record["state"],
                record["evaluation_seed"] * 1000 + state_id,
            )
            atomic_npz(
                state_path,
                initial_state=np.asarray(record["state"], dtype=np.float64),
                task_id=np.asarray(record["task_id"], dtype=np.int64),
                task_split=np.asarray(record["split"]),
                evaluation_seed=np.asarray(
                    record["evaluation_seed"], dtype=np.int64
                ),
                design_stratum=np.asarray(record["design_stratum"]),
                initial_visual=initials[0],
                initial_proprio=initial_observation["proprio"],
                selected_actions=actions,
                selected_library_indices=selected_indices,
                action_labels=np.asarray(action_labels),
                future_visual=np.asarray(visuals, dtype=np.uint8),
                future_proprio=np.asarray(proprios, dtype=np.float32),
                endpoint_states=endpoint_array,
                physical_cost=physical_cost,
                interactions=np.asarray(interactions, dtype=np.int32),
                interaction_types=np.asarray(interaction_types),
            )
            state_matrix.append(record["state"])
            task_ids.append(record["task_id"])
            action_bank.append(actions)
            physical_costs.append(physical_cost)
            interaction_bank.append(interactions)
            interaction_type_bank.append(interaction_types)
            write_json(
                OUT / f"{environment.lower()}_simulator_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "environment": environment,
                    "completed_states": state_id + 1,
                    "total_states": NUM_STATES,
                    "last_file": state_path.name,
                },
            )
            log.info(
                "%s simulator state %d/%d",
                environment,
                state_id + 1,
                NUM_STATES,
            )

        physical_costs = np.asarray(physical_costs, dtype=np.float64)
        interactions = np.asarray(interaction_bank, dtype=np.int32)
        oracle = np.argmin(physical_costs, axis=1)
        spread = np.max(physical_costs, axis=1) - np.min(
            physical_costs, axis=1
        )
        no_op_regret = physical_costs[:, 0] - np.min(
            physical_costs, axis=1
        )
        left, right = pair_indices(ACTIONS_PER_STATE)
        pair_interactions = (
            (interactions[:, left, :] > 0).astype(int)
            + (interactions[:, right, :] > 0).astype(int)
        )
        environment_design = {
            "selection_protocol": (
                "fixed state/task-relative candidates; future simulator outcomes "
                "are never used for candidate selection"
            ),
            "candidate_labels": labels,
            "no_op_oracle_fraction_by_horizon": np.mean(
                oracle == 0, axis=0
            ).tolist(),
            "no_op_positive_regret_fraction_by_horizon": np.mean(
                no_op_regret > 1e-9, axis=0
            ).tolist(),
            "median_physical_cost_spread_by_horizon": np.median(
                spread, axis=0
            ).tolist(),
            "minimum_physical_cost_spread_by_horizon": np.min(
                spread, axis=0
            ).tolist(),
            "interaction_fraction_by_horizon": np.mean(
                interactions > 0, axis=(0, 1)
            ).tolist(),
            "pair_interaction_counts": {
                label: int(np.sum(pair_interactions == index))
                for index, label in enumerate(["neither", "one", "both"])
            },
        }
        environment_design["validity_thresholds"] = {
            "final_horizon_no_op_oracle_fraction_max": 0.25,
            "final_horizon_no_op_positive_regret_fraction_min": 0.75,
            "final_horizon_median_cost_spread_min": (
                0.08 if environment == "PushT" else 0.05
            ),
            "all_pair_interaction_strata_required": True,
        }
        environment_design["design_valid"] = bool(
            environment_design[
                "no_op_oracle_fraction_by_horizon"
            ][-1]
            < 0.25
            and environment_design[
                "no_op_positive_regret_fraction_by_horizon"
            ][-1]
            > 0.75
            and environment_design[
                "median_physical_cost_spread_by_horizon"
            ][-1]
            > (0.08 if environment == "PushT" else 0.05)
            and all(
                environment_design["pair_interaction_counts"][label] > 0
                for label in ["neither", "one", "both"]
            )
        )
        design_payload[environment] = environment_design
        atomic_npz(
            OUT / f"{environment.lower()}_design.npz",
            states=np.asarray(state_matrix),
            task_ids=np.asarray(task_ids),
            action_bank=np.asarray(action_bank, dtype=np.float32),
            physical_cost=physical_costs.astype(np.float32),
            interactions=interactions,
            interaction_types=np.asarray(interaction_type_bank),
            candidate_labels=np.asarray(labels),
        )

    write_json(OUT / "tasks.json", task_payload)
    write_json(OUT / "split_manifest.json", split_payload)
    write_json(OUT / "restore_test.json", restore_payload)
    write_json(OUT / "candidate_design_summary.json", design_payload)
    return repo


if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")


In [ ]:
# Phase B — frozen checkpoint evaluation and compact feature retention.
def evaluate_models():
    repo = configure_repo()
    checkpoint_records = []
    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            model_dir = MODEL_ROOT / model_name
            model_dir.mkdir(parents=True, exist_ok=True)
            torch.cuda.reset_peak_memory_stats()
            gpu_report(f"{model_name}_before_load")
            model, preprocessor = torch.hub.load(
                str(repo),
                model_name,
                source="local",
                pretrained=True,
                device="cuda:0",
                trust_repo=True,
            )
            model.eval()
            gpu_report(f"{model_name}_after_load")

            goal_features = {}
            with torch.inference_mode():
                for task in TASKS[environment]:
                    observation = goal_observation(repo, environment, task)
                    encoded = model.encode(
                        to_model_observation(
                            observation["visual"],
                            observation["proprio"],
                        )
                    )
                    goal_features[task["task_id"]] = pool_visual(
                        encoded["visual"]
                    )[0, 0]

            horizon_index = torch.tensor(
                HORIZONS, dtype=torch.long, device="cuda"
            )
            for state_id in range(NUM_STATES):
                output_path = model_dir / f"state_{state_id:04d}.npz"
                if output_path.exists():
                    log.info(
                        "%s resume: keeping %s",
                        model_name,
                        output_path.name,
                    )
                    continue
                with np.load(
                    truth_dir / f"state_{state_id:04d}.npz"
                ) as truth:
                    task_id = int(truth["task_id"])
                    initial_visual = truth["initial_visual"]
                    initial_proprio = truth["initial_proprio"]
                    future_visual = truth["future_visual"]
                    future_proprio = truth["future_proprio"]
                    selected_actions = truth["selected_actions"]
                    physical_cost = truth["physical_cost"].astype(np.float64)

                chunks = torch.from_numpy(
                    selected_actions.reshape(
                        ACTIONS_PER_STATE,
                        max(HORIZONS),
                        FRAMESKIP,
                        2,
                    )
                ).float()
                normalized = preprocessor.normalize_actions(chunks)
                model_actions = (
                    normalized.reshape(
                        ACTIONS_PER_STATE,
                        max(HORIZONS),
                        -1,
                    )
                    .permute(1, 0, 2)
                    .contiguous()
                    .cuda()
                )
                with torch.inference_mode():
                    initial_encoded = model.encode(
                        to_model_observation(
                            initial_visual,
                            initial_proprio,
                        )
                    )
                    truth_encoded = model.encode(
                        to_model_observation(
                            future_visual,
                            future_proprio,
                        )
                    )
                    truth_features = pool_visual(truth_encoded["visual"])
                    predicted_encoded = model.unroll(
                        initial_encoded,
                        model_actions,
                    )
                    selected_prediction = predicted_encoded[
                        "visual"
                    ].index_select(0, horizon_index)
                    predicted_features = np.moveaxis(
                        pool_visual(selected_prediction),
                        0,
                        1,
                    )
                metrics = feature_metrics(
                    truth_features,
                    predicted_features,
                )
                goal_feature = goal_features[task_id]
                latent_true_cost = np.sqrt(
                    np.mean(
                        (
                            truth_features
                            - goal_feature[None, None, :]
                        )
                        ** 2,
                        axis=-1,
                    )
                )
                latent_predicted_cost = np.sqrt(
                    np.mean(
                        (
                            predicted_features
                            - goal_feature[None, None, :]
                        )
                        ** 2,
                        axis=-1,
                    )
                )
                atomic_npz(
                    output_path,
                    task_id=np.asarray(task_id),
                    predicted_features=predicted_features.astype(np.float16),
                    latent_true_cost=latent_true_cost.astype(np.float32),
                    latent_predicted_cost=latent_predicted_cost.astype(
                        np.float32
                    ),
                    physical_true_cost=physical_cost.astype(np.float32),
                    **{
                        key: np.asarray(value, dtype=np.float64)
                        for key, value in metrics.items()
                    },
                )
                write_json(
                    OUT / f"{model_name}_progress.json",
                    {
                        "run_signature": RUN_SIGNATURE,
                        "environment": environment,
                        "model": model_name,
                        "completed_states": state_id + 1,
                        "total_states": NUM_STATES,
                        "last_file": output_path.name,
                    },
                )
                log.info(
                    "%s state %d/%d",
                    model_name,
                    state_id + 1,
                    NUM_STATES,
                )
                if (state_id + 1) % 25 == 0:
                    gpu_report(f"{model_name}_state_{state_id:04d}")

            del model, preprocessor
            gc.collect()
            torch.cuda.empty_cache()
            gpu_report(f"{model_name}_released")

    hf_root = Path(os.environ["HF_HOME"]) / "hub"
    torch_root = Path(os.environ["TORCH_HOME"])
    for root in [hf_root, torch_root]:
        if root.exists():
            for path in root.rglob("*"):
                if (
                    path.is_file()
                    and path.stat().st_size > 20_000_000
                    and path.suffix
                    in {".tar", ".pth", ".pt", ".bin", ".safetensors"}
                ):
                    checkpoint_records.append(
                        {
                            "path": str(path),
                            "size_bytes": path.stat().st_size,
                            "sha256": sha256_file(path),
                        }
                    )
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "repository_commit": REPO_COMMIT,
            "training_seeds_per_public_configuration": 1,
            "training_seed_limitation": (
                "No additional public training-seed replicas are exposed by "
                "the checkpoint registry at the pinned commit."
            ),
            "cached_files": checkpoint_records,
        },
    )


if not PIPELINE_FAILED:
    try:
        evaluate_models()
    except Exception:
        record_failure("model_evaluation")


In [ ]:
# Phase C — task-disjoint probes, paired metrics, held-out regression, and decision.
def load_truth_arrays(environment):
    truth_dir = TRUTH_ROOT / environment.lower()
    endpoints = []
    costs = []
    interactions = []
    interaction_types = []
    task_ids = []
    splits = []
    evaluation_seeds = []
    for state_id in range(NUM_STATES):
        with np.load(truth_dir / f"state_{state_id:04d}.npz") as shard:
            endpoints.append(shard["endpoint_states"])
            costs.append(shard["physical_cost"])
            interactions.append(shard["interactions"])
            interaction_types.append(shard["interaction_types"])
            task_ids.append(int(shard["task_id"]))
            splits.append(str(shard["task_split"]))
            evaluation_seeds.append(int(shard["evaluation_seed"]))
    endpoints = np.asarray(endpoints, dtype=np.float64)
    return {
        "pose": pose_target(environment, endpoints),
        "physical_cost": np.asarray(costs, dtype=np.float64),
        "interactions": np.asarray(interactions, dtype=np.int32),
        "interaction_types": np.asarray(interaction_types),
        "task_id": np.asarray(task_ids, dtype=np.int64),
        "split": np.asarray(splits),
        "evaluation_seed": np.asarray(evaluation_seeds, dtype=np.int64),
    }


def load_model_arrays(model_name):
    model_dir = MODEL_ROOT / model_name
    predicted = []
    latent = []
    standards = {
        "ordinary_feature_rmse": [],
        "common_mode_feature_rmse": [],
        "action_dependent_feature_rmse": [],
        "paired_feature_rmse": [],
        "normalized_paired_feature_rmse": [],
        "paired_feature_cosine": [],
        "pair_identity_residual": [],
    }
    for state_id in range(NUM_STATES):
        with np.load(model_dir / f"state_{state_id:04d}.npz") as shard:
            predicted.append(
                shard["predicted_features"].astype(np.float32)
            )
            latent.append(
                shard["latent_predicted_cost"].astype(np.float64)
            )
            for key in standards:
                standards[key].append(
                    shard[key].astype(np.float64)
                )
    return {
        "predicted_features": np.asarray(predicted, dtype=np.float32),
        "latent_cost": np.asarray(latent, dtype=np.float64),
        **{
            key: np.asarray(values, dtype=np.float64)
            for key, values in standards.items()
        },
    }


def state_indices(truth, split_name):
    return np.flatnonzero(truth["split"] == split_name)


def flatten_states(values, indices):
    selected = np.asarray(values)[indices]
    return selected.reshape(-1, selected.shape[-1])


def fit_probe(environment, model_name, truth, model_arrays, probe_seed):
    raw = model_arrays["predicted_features"]
    input_dim = raw.shape[-1]
    projection_seed = (
        probe_seed
        + 10000 * ENVIRONMENT.index(environment)
        + 100 * MODEL_NAME.index(model_name)
    )
    projection = random_projection(
        input_dim,
        READOUT_PROJECTION_DIM,
        projection_seed,
    )
    projected = (
        raw.reshape(-1, input_dim) @ projection
    ).reshape(*raw.shape[:-1], READOUT_PROJECTION_DIM)
    train_indices = state_indices(truth, "probe_train")
    calibration_indices = state_indices(
        truth, "probe_calibration"
    )
    probe = fit_linear_readout(
        flatten_states(projected, train_indices),
        flatten_states(truth["pose"], train_indices),
        flatten_states(projected, calibration_indices),
        flatten_states(truth["pose"], calibration_indices),
    )
    probe_path = (
        PROBE_DIR
        / f"{model_name}_seed{probe_seed}_linear_pose.npz"
    )
    atomic_npz(
        probe_path,
        mean=probe["mean"],
        scale=probe["scale"],
        coefficient=probe["coefficient"],
        ridge=np.asarray(probe["ridge"]),
        projection_seed=np.asarray(projection_seed),
    )
    return probe, projection, projected, projection_seed


def evaluate_probe(
    environment,
    model_name,
    truth,
    model_arrays,
    probe,
    projected,
    probe_seed,
):
    tasks_by_id = {
        item["task_id"]: item for item in TASKS[environment]
    }
    evaluation_indices = np.concatenate(
        [
            state_indices(truth, "regression_train"),
            state_indices(truth, "final_test"),
        ]
    )
    prediction = predict_linear_readout(
        probe,
        flatten_states(projected, evaluation_indices),
    ).reshape(
        len(evaluation_indices),
        ACTIONS_PER_STATE,
        len(HORIZONS),
        truth["pose"].shape[-1],
    )

    unit_rows = []
    pair_rows = []
    action_rows = []
    for local_index, state_id in enumerate(evaluation_indices):
        split_name = str(truth["split"][state_id])
        task_id = int(truth["task_id"][state_id])
        task = tasks_by_id[task_id]
        true_pose = truth["pose"][state_id]
        true_cost = truth["physical_cost"][state_id]
        interactions = truth["interactions"][state_id]
        interaction_types = truth["interaction_types"][state_id]
        linear_pose = prediction[local_index]
        linear_cost = decoded_task_cost(
            environment,
            linear_pose,
            task,
        )
        latent_cost = model_arrays["latent_cost"][state_id]
        readout_costs = {
            "latent_distance": latent_cost,
            "linear_pose": linear_cost,
            "action_blind": np.zeros_like(true_cost),
            "linear_pose_shuffled": np.roll(
                linear_cost, 1, axis=0
            ),
            "oracle_pose": true_cost.copy(),
        }
        for horizon_index, horizon in enumerate(HORIZONS):
            truth_cost_vector = true_cost[:, horizon_index]
            truth_pose_vector = true_pose[:, horizon_index]
            interaction_vector = interactions[:, horizon_index]
            interaction_type_vector = interaction_types[:, horizon_index]
            standard_payload = {
                key: float(model_arrays[key][state_id, horizon_index])
                for key in [
                    "ordinary_feature_rmse",
                    "common_mode_feature_rmse",
                    "action_dependent_feature_rmse",
                    "paired_feature_rmse",
                    "normalized_paired_feature_rmse",
                    "paired_feature_cosine",
                    "pair_identity_residual",
                ]
            }
            for readout in READOUTS:
                predicted_cost_vector = readout_costs[readout][
                    :, horizon_index
                ]
                ranking = ranking_metrics(
                    truth_cost_vector,
                    predicted_cost_vector,
                )
                if readout == "linear_pose":
                    predicted_pose_vector = linear_pose[
                        :, horizon_index
                    ]
                    pose_error = float(
                        np.mean(
                            physical_pose_error(
                                environment,
                                predicted_pose_vector,
                                truth_pose_vector,
                            )
                        )
                    )
                    physical_cost_rmse = float(
                        np.sqrt(
                            np.mean(
                                (
                                    predicted_cost_vector
                                    - truth_cost_vector
                                )
                                ** 2
                            )
                        )
                    )
                else:
                    predicted_pose_vector = None
                    pose_error = float("nan")
                    physical_cost_rmse = float("nan")
                unit_rows.append(
                    {
                        "environment": environment,
                        "state_id": int(state_id),
                        "task_id": task_id,
                        "task_name": task["task_name"],
                        "split": split_name,
                        "evaluation_seed": int(
                            truth["evaluation_seed"][state_id]
                        ),
                        "model": model_name,
                        "model_family": (
                            "DINO-WM"
                            if model_name.startswith("dino")
                            else "JEPA-WM"
                        ),
                        "probe_seed": int(probe_seed),
                        "readout": readout,
                        "horizon": int(horizon),
                        "pose_error": pose_error,
                        "physical_cost_rmse": physical_cost_rmse,
                        "top1_correct": ranking["top1_correct"],
                        "regret": ranking["regret"],
                        "normalized_regret": ranking[
                            "normalized_regret"
                        ],
                        "pairwise_accuracy": ranking[
                            "pairwise_accuracy"
                        ],
                        "weighted_pairwise_accuracy": ranking[
                            "weighted_pairwise_accuracy"
                        ],
                        "normalized_margin_rmse": ranking[
                            "normalized_margin_rmse"
                        ],
                        "interaction_fraction": float(
                            np.mean(interaction_vector > 0)
                        ),
                        "selected_action": ranking[
                            "selected_action"
                        ],
                        "oracle_action": ranking["oracle_action"],
                        **standard_payload,
                    }
                )

                if split_name != "final_test":
                    continue
                for action_id in range(ACTIONS_PER_STATE):
                    row = {
                        "environment": environment,
                        "state_id": int(state_id),
                        "task_id": task_id,
                        "model": model_name,
                        "probe_seed": int(probe_seed),
                        "readout": readout,
                        "horizon": int(horizon),
                        "action": int(action_id),
                        "interaction_type": str(
                            interaction_type_vector[action_id]
                        ),
                        "true_cost": float(
                            truth_cost_vector[action_id]
                        ),
                        "predicted_cost": float(
                            predicted_cost_vector[action_id]
                        ),
                        "true_pose_0": float("nan"),
                        "true_pose_1": float("nan"),
                        "true_pose_2": float("nan"),
                        "true_pose_3": float("nan"),
                        "predicted_pose_0": float("nan"),
                        "predicted_pose_1": float("nan"),
                        "predicted_pose_2": float("nan"),
                        "predicted_pose_3": float("nan"),
                    }
                    if predicted_pose_vector is not None:
                        for target_index in range(
                            predicted_pose_vector.shape[-1]
                        ):
                            row[f"true_pose_{target_index}"] = float(
                                truth_pose_vector[
                                    action_id, target_index
                                ]
                            )
                            row[
                                f"predicted_pose_{target_index}"
                            ] = float(
                                predicted_pose_vector[
                                    action_id, target_index
                                ]
                            )
                    action_rows.append(row)

                for pair_index, (left, right) in enumerate(
                    zip(
                        ranking["pair_left"],
                        ranking["pair_right"],
                    )
                ):
                    interaction_count = int(
                        interaction_vector[left] > 0
                    ) + int(interaction_vector[right] > 0)
                    pair_rows.append(
                        {
                            "environment": environment,
                            "state_id": int(state_id),
                            "task_id": task_id,
                            "model": model_name,
                            "probe_seed": int(probe_seed),
                            "readout": readout,
                            "horizon": int(horizon),
                            "pair_left": int(left),
                            "pair_right": int(right),
                            "interaction_stratum": [
                                "neither",
                                "one",
                                "both",
                            ][interaction_count],
                            "left_interaction_type": str(
                                interaction_type_vector[left]
                            ),
                            "right_interaction_type": str(
                                interaction_type_vector[right]
                            ),
                            "true_margin": float(
                                ranking["true_margin"][pair_index]
                            ),
                            "predicted_margin": float(
                                ranking["predicted_margin"][
                                    pair_index
                                ]
                            ),
                            "ranking_credit": float(
                                ranking["pair_credit"][pair_index]
                            ),
                            "margin_weight": float(
                                ranking["pair_weight"][pair_index]
                            ),
                        }
                    )
    return unit_rows, pair_rows, action_rows


def summarize_final(unit_rows):
    metrics = [
        "pose_error",
        "physical_cost_rmse",
        "top1_correct",
        "normalized_regret",
        "pairwise_accuracy",
        "weighted_pairwise_accuracy",
        "normalized_margin_rmse",
        "ordinary_feature_rmse",
        "normalized_paired_feature_rmse",
    ]
    rows = []
    final = [row for row in unit_rows if row["split"] == "final_test"]
    for environment in ENVIRONMENT:
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            for readout in READOUTS:
                for horizon in HORIZONS:
                    selected = [
                        row
                        for row in final
                        if row["environment"] == environment
                        and row["model"] == model_name
                        and row["readout"] == readout
                        and row["horizon"] == horizon
                    ]
                    groups = np.asarray(
                        [
                            f"{row['environment']}:{row['state_id']}"
                            for row in selected
                        ]
                    )
                    summary = {
                        "environment": environment,
                        "model": model_name,
                        "readout": readout,
                        "horizon": horizon,
                        "num_final_rows": len(selected),
                        "num_final_states": len(
                            set(
                                row["state_id"]
                                for row in selected
                            )
                        ),
                        "num_probe_seeds": len(
                            set(
                                row["probe_seed"]
                                for row in selected
                            )
                        ),
                    }
                    for metric_index, metric in enumerate(metrics):
                        interval = bootstrap_mean(
                            [row[metric] for row in selected],
                            groups,
                            BOOTSTRAP_REPS,
                            SEED
                            + 1000 * ENVIRONMENT.index(environment)
                            + 100 * MODEL_NAME.index(model_name)
                            + 10 * READOUTS.index(readout)
                            + metric_index
                            + horizon,
                        )
                        summary[metric] = interval["estimate"]
                        summary[f"{metric}_low"] = interval["low"]
                        summary[f"{metric}_high"] = interval["high"]
                    rows.append(summary)
    return rows


def paired_improvement(
    unit_rows,
    environment,
    readout,
    metric,
    higher_is_better,
):
    final = [
        row
        for row in unit_rows
        if row["split"] == "final_test"
        and (
            environment == "pooled"
            or row["environment"] == environment
        )
    ]
    baseline = {
        (
            row["environment"],
            row["state_id"],
            row["model"],
            row["horizon"],
            row["probe_seed"],
        ): row[metric]
        for row in final
        if row["readout"] == "latent_distance"
    }
    candidate = {
        (
            row["environment"],
            row["state_id"],
            row["model"],
            row["horizon"],
            row["probe_seed"],
        ): row[metric]
        for row in final
        if row["readout"] == readout
    }
    keys = sorted(set(baseline) & set(candidate))
    if higher_is_better:
        values = np.asarray(
            [candidate[key] - baseline[key] for key in keys]
        )
    else:
        values = np.asarray(
            [baseline[key] - candidate[key] for key in keys]
        )
    groups = np.asarray([f"{key[0]}:{key[1]}" for key in keys])
    seed_offset = (
        0 if environment == "pooled" else 100 * ENVIRONMENT.index(environment)
    )
    return bootstrap_mean(
        values,
        groups,
        BOOTSTRAP_REPS,
        SEED + 5000 + seed_offset + READOUTS.index(readout),
    )


def regression_design(rows, include_feature_pair, include_task_pair):
    values = []
    for row in rows:
        vector = [
            row["pose_error"],
            row["physical_cost_rmse"],
            row["ordinary_feature_rmse"],
            row["common_mode_feature_rmse"],
            row["interaction_fraction"],
            float(row["environment"] == "Wall"),
            float(row["model_family"] == "JEPA-WM"),
            float(row["horizon"] == 3),
            float(row["horizon"] == 6),
        ]
        if include_feature_pair:
            vector.append(row["normalized_paired_feature_rmse"])
        if include_task_pair:
            vector.append(row["normalized_margin_rmse"])
        values.append(vector)
    return np.asarray(values, dtype=np.float64)


def fit_ridge_regression(x, y, ridge=REGRESSION_RIDGE):
    mean, scale = standardize_fit(x)
    standardized = (x - mean) / scale
    augmented = np.column_stack(
        [np.ones(len(standardized)), standardized]
    )
    penalty = np.eye(augmented.shape[1]) * ridge
    penalty[0, 0] = 0.0
    coefficient = np.linalg.solve(
        augmented.T @ augmented + penalty,
        augmented.T @ y,
    )
    return {
        "mean": mean,
        "scale": scale,
        "coefficient": coefficient,
    }


def predict_ridge_regression(model, x):
    standardized = (x - model["mean"]) / model["scale"]
    augmented = np.column_stack(
        [np.ones(len(standardized)), standardized]
    )
    return augmented @ model["coefficient"]


def regression_metrics(y, prediction):
    y = np.asarray(y, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    mae = float(np.mean(np.abs(y - prediction)))
    denominator = float(np.sum((y - np.mean(y)) ** 2))
    r2 = (
        1.0
        - float(np.sum((y - prediction) ** 2)) / denominator
        if denominator > 1e-12
        else float("nan")
    )
    return {"mae": mae, "r2": r2}


def held_out_regression(unit_rows):
    linear = [
        row
        for row in unit_rows
        if row["readout"] == "linear_pose"
    ]
    train = [
        row for row in linear if row["split"] == "regression_train"
    ]
    test = [row for row in linear if row["split"] == "final_test"]
    y_train = np.asarray(
        [row["normalized_regret"] for row in train],
        dtype=np.float64,
    )
    y_test = np.asarray(
        [row["normalized_regret"] for row in test],
        dtype=np.float64,
    )
    specifications = {
        "ordinary_only": (False, False),
        "ordinary_plus_raw_feature_pair": (True, False),
        "ordinary_plus_task_margin_pair": (False, True),
    }
    payload = {
        "protocol": (
            "fixed-ridge regression trained on regression-train tasks/states "
            "and evaluated once on disjoint final-test tasks/states"
        ),
        "outcome": "normalized_regret",
        "models": {},
    }
    predictions = {}
    for name, (feature_pair, task_pair) in specifications.items():
        x_train = regression_design(
            train,
            include_feature_pair=feature_pair,
            include_task_pair=task_pair,
        )
        x_test = regression_design(
            test,
            include_feature_pair=feature_pair,
            include_task_pair=task_pair,
        )
        model = fit_ridge_regression(x_train, y_train)
        prediction = predict_ridge_regression(model, x_test)
        predictions[name] = prediction
        payload["models"][name] = {
            **regression_metrics(y_test, prediction),
            "num_train_rows": len(train),
            "num_test_rows": len(test),
            "num_train_state_clusters": len(
                set(
                    f"{row['environment']}:{row['state_id']}"
                    for row in train
                )
            ),
            "num_test_state_clusters": len(
                set(
                    f"{row['environment']}:{row['state_id']}"
                    for row in test
                )
            ),
        }
    baseline_error = np.abs(
        y_test - predictions["ordinary_only"]
    )
    groups = np.asarray(
        [
            f"{row['environment']}:{row['state_id']}"
            for row in test
        ]
    )
    for name in [
        "ordinary_plus_raw_feature_pair",
        "ordinary_plus_task_margin_pair",
    ]:
        candidate_error = np.abs(y_test - predictions[name])
        payload[f"{name}_mae_improvement"] = bootstrap_mean(
            baseline_error - candidate_error,
            groups,
            BOOTSTRAP_REPS,
            SEED
            + 8000
            + list(specifications).index(name),
        )
    prediction_rows = []
    for index, row in enumerate(test):
        prediction_rows.append(
            {
                "environment": row["environment"],
                "state_id": row["state_id"],
                "task_id": row["task_id"],
                "model": row["model"],
                "probe_seed": row["probe_seed"],
                "horizon": row["horizon"],
                "normalized_regret": y_test[index],
                "ordinary_prediction": predictions[
                    "ordinary_only"
                ][index],
                "ordinary_plus_raw_pair_prediction": predictions[
                    "ordinary_plus_raw_feature_pair"
                ][index],
                "ordinary_plus_task_pair_prediction": predictions[
                    "ordinary_plus_task_margin_pair"
                ][index],
            }
        )
    return payload, prediction_rows


def rank_values(values, higher_is_better=False):
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(-values if higher_is_better else values)
    ranks = np.empty(len(values), dtype=int)
    ranks[order] = np.arange(1, len(values) + 1)
    return ranks


def build_model_rankings(unit_rows):
    final_linear = [
        row
        for row in unit_rows
        if row["split"] == "final_test"
        and row["readout"] == "linear_pose"
    ]
    rows = []
    for environment in ENVIRONMENT:
        for horizon in HORIZONS:
            aggregate = []
            for model_name in MODEL_BY_ENVIRONMENT[environment]:
                selected = [
                    row
                    for row in final_linear
                    if row["environment"] == environment
                    and row["horizon"] == horizon
                    and row["model"] == model_name
                ]
                aggregate.append(
                    {
                        "environment": environment,
                        "horizon": horizon,
                        "model": model_name,
                        "ordinary_cost_rmse": float(
                            np.mean(
                                [
                                    row["physical_cost_rmse"]
                                    for row in selected
                                ]
                            )
                        ),
                        "counterfactual_margin_rmse": float(
                            np.mean(
                                [
                                    row["normalized_margin_rmse"]
                                    for row in selected
                                ]
                            )
                        ),
                        "weighted_pairwise_accuracy": float(
                            np.mean(
                                [
                                    row[
                                        "weighted_pairwise_accuracy"
                                    ]
                                    for row in selected
                                ]
                            )
                        ),
                        "normalized_regret": float(
                            np.mean(
                                [
                                    row["normalized_regret"]
                                    for row in selected
                                ]
                            )
                        ),
                    }
                )
            ordinary_ranks = rank_values(
                [item["ordinary_cost_rmse"] for item in aggregate]
            )
            counterfactual_ranks = rank_values(
                [
                    item["counterfactual_margin_rmse"]
                    for item in aggregate
                ]
            )
            planning_ranks = rank_values(
                [item["normalized_regret"] for item in aggregate]
            )
            reversal = not np.array_equal(
                ordinary_ranks, counterfactual_ranks
            )
            for index, item in enumerate(aggregate):
                rows.append(
                    {
                        **item,
                        "ordinary_rank": int(ordinary_ranks[index]),
                        "counterfactual_rank": int(
                            counterfactual_ranks[index]
                        ),
                        "planning_rank": int(planning_ranks[index]),
                        "ordinary_vs_counterfactual_reversal": reversal,
                    }
                )
    return rows


def decision_payload(unit_rows, regression):
    environments = {}
    for environment in [*ENVIRONMENT, "pooled"]:
        environments[environment] = {
            "normalized_regret_improvement": paired_improvement(
                unit_rows,
                environment,
                "linear_pose",
                "normalized_regret",
                False,
            ),
            "weighted_pairwise_accuracy_improvement": paired_improvement(
                unit_rows,
                environment,
                "linear_pose",
                "weighted_pairwise_accuracy",
                True,
            ),
            "top1_accuracy_improvement": paired_improvement(
                unit_rows,
                environment,
                "linear_pose",
                "top1_correct",
                True,
            ),
        }
    environment_pass = {
        environment: (
            environments[environment][
                "normalized_regret_improvement"
            ]["low"]
            > 0
            and environments[environment][
                "weighted_pairwise_accuracy_improvement"
            ]["low"]
            > 0
        )
        for environment in ENVIRONMENT
    }
    regression_pass = (
        regression[
            "ordinary_plus_task_margin_pair_mae_improvement"
        ]["low"]
        > 0
    )
    design = json.loads(
        (OUT / "candidate_design_summary.json").read_text()
    )
    restore = json.loads((OUT / "restore_test.json").read_text())
    integrity_pass = bool(
        all(design[environment]["design_valid"] for environment in ENVIRONMENT)
        and all(
            restore[environment]["endpoint_bitwise_exact"]
            and restore[environment]["initial_render_bitwise_exact"]
            and restore[environment]["diagnostics_exact"]
            for environment in ENVIRONMENT
        )
    )
    if not integrity_pass:
        status = "INCONCLUSIVE"
    elif all(environment_pass.values()) and regression_pass:
        status = "CROSS_ENV_TASK_ALIGNED_SIGNAL"
    elif all(environment_pass.values()):
        status = "CROSS_ENV_PLANNING_SIGNAL_ONLY"
    else:
        status = "MIXED_GENERALIZATION"
    return {
        "status": status,
        "primary_readout": "linear_pose",
        "baseline": "latent_distance",
        "environment_comparisons": environments,
        "environment_gate_pass": environment_pass,
        "held_out_regression_gate_pass": regression_pass,
        "design_and_restore_integrity_pass": integrity_pass,
        "gate": (
            "Candidate-design and exact-restore integrity must pass. Both "
            "environments then require state-clustered 95% bootstrap lower "
            "bounds above zero for regret and weighted-ranking improvement. "
            "The full signal additionally requires a positive lower bound for "
            "held-out MAE improvement from task-margin counterfactual error."
        ),
        "interpretation_guardrail": (
            "This licenses simulator-generalization claims only; it does not "
            "establish real-robot reliability."
        ),
    }


def make_plots(summary_rows, regression_rows, ranking_rows):
    display_readouts = [
        "latent_distance",
        "linear_pose",
        "action_blind",
        "linear_pose_shuffled",
        "oracle_pose",
    ]
    labels = ["latent", "linear", "blind", "shuffled", "oracle"]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
    for row_index, environment in enumerate(ENVIRONMENT):
        for column_index, horizon in enumerate(HORIZONS):
            axis = axes[row_index, column_index]
            values = []
            for readout in display_readouts:
                selected = [
                    row["normalized_regret"]
                    for row in summary_rows
                    if row["environment"] == environment
                    and row["horizon"] == horizon
                    and row["readout"] == readout
                ]
                values.append(float(np.mean(selected)))
            axis.bar(labels, values)
            axis.set_title(f"{environment}, horizon {horizon}")
            axis.tick_params(axis="x", rotation=30)
            if column_index == 0:
                axis.set_ylabel("physical normalized regret")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "regret_by_environment.png", dpi=180)
    plt.close(fig)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
    for row_index, environment in enumerate(ENVIRONMENT):
        for column_index, horizon in enumerate(HORIZONS):
            axis = axes[row_index, column_index]
            values = []
            for readout in display_readouts:
                selected = [
                    row["weighted_pairwise_accuracy"]
                    for row in summary_rows
                    if row["environment"] == environment
                    and row["horizon"] == horizon
                    and row["readout"] == readout
                ]
                values.append(float(np.mean(selected)))
            axis.bar(labels, values)
            axis.axhline(0.5, color="black", linewidth=1)
            axis.set_ylim(0, 1)
            axis.set_title(f"{environment}, horizon {horizon}")
            axis.tick_params(axis="x", rotation=30)
            if column_index == 0:
                axis.set_ylabel("margin-weighted pair accuracy")
    fig.tight_layout()
    fig.savefig(
        PLOT_DIR / "weighted_ranking_by_environment.png",
        dpi=180,
    )
    plt.close(fig)

    truth = np.asarray(
        [row["normalized_regret"] for row in regression_rows]
    )
    ordinary = np.asarray(
        [row["ordinary_prediction"] for row in regression_rows]
    )
    task_pair = np.asarray(
        [
            row["ordinary_plus_task_pair_prediction"]
            for row in regression_rows
        ]
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for axis, prediction, title in [
        (axes[0], ordinary, "ordinary-only"),
        (axes[1], task_pair, "ordinary + task-pair"),
    ]:
        axis.scatter(truth, prediction, s=8, alpha=0.25)
        axis.plot([0, 1], [0, 1], color="black")
        axis.set_xlim(-0.05, 1.05)
        axis.set_ylim(-0.05, 1.05)
        axis.set_xlabel("true normalized regret")
        axis.set_ylabel("held-out prediction")
        axis.set_title(title)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "held_out_regression.png", dpi=180)
    plt.close(fig)

    reversal_count = sum(
        bool(row["ordinary_vs_counterfactual_reversal"])
        for row in ranking_rows[::2]
    )
    fig, axis = plt.subplots(figsize=(7, 4))
    groups = [
        f"{row['environment']}-H{row['horizon']}"
        for row in ranking_rows[::2]
    ]
    values = [
        int(row["ordinary_vs_counterfactual_reversal"])
        for row in ranking_rows[::2]
    ]
    axis.bar(groups, values)
    axis.set_ylim(0, 1.2)
    axis.set_ylabel("ordinary/counterfactual rank reversal")
    axis.set_title(f"Rank reversals: {reversal_count}/{len(values)}")
    axis.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "model_rank_reversals.png", dpi=180)
    plt.close(fig)


def run_analysis():
    unit_rows = []
    pair_rows = []
    action_rows = []
    probe_manifest = {
        "world_models_frozen": True,
        "test_states_used_for_fitting": False,
        "task_disjoint": True,
        "target_by_environment": {
            "PushT": [
                "block_x_over_512",
                "block_y_over_512",
                "sin_theta",
                "cos_theta",
            ],
            "Wall": ["dot_x_over_65", "dot_y_over_65"],
        },
        "projection_dim": READOUT_PROJECTION_DIM,
        "hyperparameter_selection": "probe-calibration pose MSE only",
        "probe_seeds": PROBE_SEEDS,
        "models": {},
    }

    for environment in ENVIRONMENT:
        truth = load_truth_arrays(environment)
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            model_arrays = load_model_arrays(model_name)
            probe_manifest["models"][model_name] = {}
            for probe_seed in PROBE_SEEDS:
                probe, projection, projected, projection_seed = fit_probe(
                    environment,
                    model_name,
                    truth,
                    model_arrays,
                    probe_seed,
                )
                probe_manifest["models"][model_name][str(probe_seed)] = {
                    "raw_feature_dim": int(
                        model_arrays["predicted_features"].shape[-1]
                    ),
                    "projection_seed": int(projection_seed),
                    "linear_ridge": probe["ridge"],
                    "linear_calibration_pose_mse": probe[
                        "calibration_pose_mse"
                    ],
                }
                model_unit, model_pair, model_action = evaluate_probe(
                    environment,
                    model_name,
                    truth,
                    model_arrays,
                    probe,
                    projected,
                    probe_seed,
                )
                unit_rows.extend(model_unit)
                pair_rows.extend(model_pair)
                action_rows.extend(model_action)
                del projection, projected
                gc.collect()
            del model_arrays
            gc.collect()
            gpu_report(f"{model_name}_probe_complete")

    summary_rows = summarize_final(unit_rows)
    regression, regression_rows = held_out_regression(unit_rows)
    ranking_rows = build_model_rankings(unit_rows)
    decision = decision_payload(unit_rows, regression)

    write_csv(OUT / "unit_metrics.csv", unit_rows)
    write_csv(OUT / "pair_metrics.csv", pair_rows)
    write_csv(OUT / "action_predictions.csv", action_rows)
    write_csv(OUT / "metrics_summary.csv", summary_rows)
    write_csv(OUT / "held_out_regression_predictions.csv", regression_rows)
    write_csv(OUT / "model_rankings.csv", ranking_rows)
    write_json(OUT / "probe_manifest.json", probe_manifest)
    write_json(OUT / "held_out_regression.json", regression)
    write_json(OUT / "stage3_decision.json", decision)
    write_json(
        OUT / "metrics_summary.json",
        {
            "status": "SUCCESS",
            "run_mode": RUN_MODE,
            "num_states_per_environment": NUM_STATES,
            "environments": ENVIRONMENT,
            "models": MODEL_NAME,
            "horizons": HORIZONS,
            "readouts": READOUTS,
            "probe_seeds": PROBE_SEEDS,
            "decision": decision,
            "summary_rows": summary_rows,
        },
    )

    make_plots(summary_rows, regression_rows, ranking_rows)

    # Structural invariants before any success label is written.
    expected_splits = {
        "probe_train": TASK_SPLIT_COUNTS[0]
        * (NUM_STATES // TASKS_PER_ENVIRONMENT),
        "probe_calibration": TASK_SPLIT_COUNTS[1]
        * (NUM_STATES // TASKS_PER_ENVIRONMENT),
        "regression_train": TASK_SPLIT_COUNTS[2]
        * (NUM_STATES // TASKS_PER_ENVIRONMENT),
        "final_test": TASK_SPLIT_COUNTS[3]
        * (NUM_STATES // TASKS_PER_ENVIRONMENT),
    }
    for environment in ENVIRONMENT:
        truth = load_truth_arrays(environment)
        observed = {
            name: int(np.sum(truth["split"] == name))
            for name in SPLIT_NAMES
        }
        if observed != expected_splits:
            raise AssertionError(
                f"{environment} split mismatch: {observed}"
            )
    expected_unit_rows = (
        len(ENVIRONMENT)
        * (expected_splits["regression_train"] + expected_splits["final_test"])
        * 2
        * len(PROBE_SEEDS)
        * len(HORIZONS)
        * len(READOUTS)
    )
    expected_pair_rows = (
        len(ENVIRONMENT)
        * expected_splits["final_test"]
        * 2
        * len(PROBE_SEEDS)
        * len(HORIZONS)
        * len(READOUTS)
        * math.comb(ACTIONS_PER_STATE, 2)
    )
    expected_action_rows = (
        len(ENVIRONMENT)
        * expected_splits["final_test"]
        * 2
        * len(PROBE_SEEDS)
        * len(HORIZONS)
        * len(READOUTS)
        * ACTIONS_PER_STATE
    )
    if len(unit_rows) != expected_unit_rows:
        raise AssertionError(
            f"unit row count {len(unit_rows)} != {expected_unit_rows}"
        )
    if len(pair_rows) != expected_pair_rows:
        raise AssertionError(
            f"pair row count {len(pair_rows)} != {expected_pair_rows}"
        )
    if len(action_rows) != expected_action_rows:
        raise AssertionError(
            f"action row count {len(action_rows)} != {expected_action_rows}"
        )
    if any(
        row["split"] == "probe_train"
        or row["split"] == "probe_calibration"
        for row in unit_rows
    ):
        raise AssertionError("probe-fit rows leaked into evaluation tables")
    return decision


if not PIPELINE_FAILED:
    try:
        DECISION = run_analysis()
        (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
        gpu_report("analysis_complete")
    except Exception:
        record_failure("analysis")


In [ ]:
def package_results():
    include_names = [
        "FAILURE_TRACE.txt",
        "action_predictions.csv",
        "candidate_design_summary.json",
        "checkpoints_manifest.json",
        "config.json",
        "held_out_regression.json",
        "held_out_regression_predictions.csv",
        "model_rankings.csv",
        "metrics_summary.csv",
        "metrics_summary.json",
        "pair_metrics.csv",
        "probe_manifest.json",
        "restore_test.json",
        "split_manifest.json",
        "stage3_decision.json",
        "tasks.json",
        "versions.json",
        "pusht_design.npz",
        "wall_design.npz",
        "plots/regret_by_environment.png",
        "plots/weighted_ranking_by_environment.png",
        "plots/held_out_regression.png",
        "plots/model_rank_reversals.png",
        "logs/run.log",
    ]
    include_names.extend(
        path.relative_to(OUT).as_posix()
        for path in sorted(PROBE_DIR.glob("*"))
    )
    include_names.extend(
        path.name
        for path in sorted(OUT.glob("*_progress.json"))
    )
    archive = Path("/content/stage3_result_bundle.zip")
    if MOUNT_DRIVE:
        archive = OUT.parent / "stage3_result_bundle.zip"
    with zipfile.ZipFile(
        archive, "w", compression=zipfile.ZIP_DEFLATED
    ) as handle:
        for name in include_names:
            path = OUT / name
            if path.exists():
                handle.write(path, arcname=name)
    write_json(
        OUT / "result_zip_manifest.json",
        {
            "archive": str(archive),
            "included": [
                name
                for name in include_names
                if (OUT / name).exists()
            ],
            "intermediate_excluded": True,
        },
    )
    with zipfile.ZipFile(
        archive, "a", compression=zipfile.ZIP_DEFLATED
    ) as handle:
        handle.write(
            OUT / "result_zip_manifest.json",
            arcname="result_zip_manifest.json",
        )
    return archive


try:
    RESULT_ZIP = package_results()
    print(f"Result bundle: {RESULT_ZIP}")
    if not MOUNT_DRIVE:
        from google.colab import files

        files.download(str(RESULT_ZIP))
        print("Automatic browser download requested.")
    else:
        print("Bundle retained in Google Drive; no browser download requested.")
except Exception:
    record_failure("packaging")
    raise


In [ ]:
if PIPELINE_FAILED:
    print("STAGE 3 FAILED")
    print(FAILURE_MESSAGE)
    print(f"Return the downloaded bundle containing {OUT / 'FAILURE_TRACE.txt'}.")
else:
    decision = json.loads((OUT / "stage3_decision.json").read_text())
    print("STAGE 3 COMPLETE")
    print(json.dumps(decision, indent=2))
    print("Return stage3_result_bundle.zip for independent audit.")
